# Estimating insect spatial resolution with the optomotor response

During this lab, we investigates whether a fly turns in response to a rotating visual grating, and whether that response depends on the **spatial frequency** of the grating.

We obtained from our experimental setup a CSV file per spatial frequency tested.
Each CSV file contains four columns:

- `frame` — video/tracking frame
- `time_s` — time in seconds
- `ball_yaw_deg` — **ball rotation between consecutive measurements**, in degrees
- `direction` — grating direction: `+1 = clockwise (CW)`, `-1 = counter-clockwise (CCW)`, `0 = stationary`

### Important sign convention
The fly walks on top of the ball, so the ball rotates in the **opposite direction** to the fly. Positive ball yaw is CW.

Therefore, when the fly follows the grating correctly:

| Grating | Fly | Expected ball rotation |
|---|---|---|
| `+1` CW | CW | CCW (negative) |
| `-1` CCW | CCW | CW (positive) |

## Data setup

In [ ]:
# ===== Retrieving dataset from GitHub =====
GITHUB_USER = "bClemot-Sc"
REPO_NAME = "neurobiology-course-lab4"
BRANCH = "main"
YEAR = "2026"

GROUP_PAIRS = ["1-2", "3-4", "5-6", "7-8", "9-10", "backup"]
# ============================================================


## 1. Load Python packages

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Packages loaded.")

## 2. Select your experimental group and load the data
Choose your group from the dropdown and click **Load group data**. The notebook will download your group's CSV files automatically.

In [ ]:
import io
import zipfile
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Make the dropdown labels nice for students
group_options = [
    ("Backup dataset" if pair == "backup" else f"Group {pair}", pair)
    for pair in GROUP_PAIRS
]

group_dropdown = widgets.Dropdown(
    options=group_options,
    description="Group:",
    value=GROUP_PAIRS[0],
)

load_button = widgets.Button(
    description="Load group data",
    button_style="primary"
)

load_output = widgets.Output()


def load_group_data(_):
    global csv_sources, selected_group

    selected_group = str(group_dropdown.value)

    # Normal groups are called group1-2.zip, group3-4.zip, etc.
    # The backup is simply called backup.zip.
    if selected_group == "backup":
        zip_name = "backup.zip"
        dataset_name = "Backup dataset"
    else:
        zip_name = f"group{selected_group}.zip"
        dataset_name = f"Group {selected_group}"

    url = (
        f"https://raw.githubusercontent.com/"
        f"{GITHUB_USER}/{REPO_NAME}/{BRANCH}/"
        f"data/{YEAR}/{zip_name}"
    )

    with load_output:
        clear_output()

        if "YOUR_GITHUB" in GITHUB_USER or "YOUR_REPOSITORY" in REPO_NAME:
            print(
                "Teacher setup is incomplete: "
                "GITHUB_USER and REPO_NAME must be configured."
            )
            return

        print(f"Loading {dataset_name} data ({YEAR})...")

        try:
            # Download ZIP from GitHub
            with urllib.request.urlopen(url) as response:
                zip_bytes = response.read()

            # Extract CSV files from ZIP
            with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
                csv_names = [
                    n for n in zf.namelist()
                    if n.lower().endswith(".csv")
                    and not n.endswith("/")
                ]

                if not csv_names:
                    raise ValueError("The dataset ZIP contains no CSV files.")

                csv_sources = {
                    Path(name).name: zf.read(name)
                    for name in csv_names
                }

            # Check that the processed CSVs have the expected columns
            required = {
                "frame",
                "time_s",
                "ball_yaw_deg",
                "direction"
            }

            problems = []

            for name, raw_bytes in csv_sources.items():
                test = pd.read_csv(
                    io.BytesIO(raw_bytes),
                    nrows=5
                )

                missing = sorted(required - set(test.columns))

                if missing:
                    problems.append(
                        f"{name}: missing {', '.join(missing)}"
                    )

            if problems:
                csv_sources = {}
                raise ValueError(
                    "Invalid processed data:\n"
                    + "\n".join(problems)
                )

            print(f"✓ {dataset_name} loaded")
            print(f"✓ {len(csv_sources)} CSV file(s) found")
            print("Ready to analyse.")

        except Exception as exc:
            csv_sources = {}
            print("Could not load the dataset.")
            print(f"Address tried: {url}")
            print(f"Reason: {exc}")


load_button.on_click(load_group_data)

display(
    widgets.HBox([group_dropdown, load_button]),
    load_output
)

## 3. Read the files and identify spatial frequency
The spatial frequency is stored in each processed filename. We extract it and combine all recordings into one table.

In [ ]:
def spatial_frequency_from_filename(filename):
    match = re.search(r"SF-([0-9.]+)", filename)
    if not match:
        raise ValueError(f"Could not find spatial frequency in filename: {filename}")
    return float(match.group(1))

if not globals().get("csv_sources"):
    raise RuntimeError("No group data are loaded. Go back to Step 2, choose your group, and click 'Load group data'.")

required = {"frame", "time_s", "ball_yaw_deg", "direction"}
recordings = []
for filename, raw_bytes in sorted(csv_sources.items()):
    df = pd.read_csv(io.BytesIO(raw_bytes))
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing required columns: {sorted(missing)}")
    df = df.copy()
    df["spatial_frequency"] = spatial_frequency_from_filename(filename)
    df["recording"] = filename
    recordings.append(df)

data = pd.concat(recordings, ignore_index=True)
frequencies = sorted(data["spatial_frequency"].unique())
print(f"Loaded {len(recordings)} recordings for Group {selected_group}.")
print("Spatial frequencies (cycles/degree):", frequencies)
data.head()


## 4. First look: what is `ball_yaw_deg`?

`ball_yaw_deg` is **not** the total angle of the ball. It is the small rotation that occurred between two consecutive measurements.

Plotting a short part of one recording makes this easier to see.

In [ ]:
example_name = data["recording"].iloc[0]
example = data[data["recording"] == example_name].copy()

plt.figure(figsize=(10, 4))
plt.plot(example["time_s"].iloc[:1500], example["ball_yaw_deg"].iloc[:1500], linewidth=0.8)
plt.axhline(0, linewidth=0.8, linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Ball yaw between samples (deg)")
plt.title(f"Frame-to-frame ball rotation\n{example_name}")
plt.show()

## 5. Identify the stimulus-direction bouts

A **bout** is one uninterrupted period with the same `direction` value. They are measurement repetitions for the same spatial frequency values. We assign a new `direction_block` whenever direction changes.

This includes the stationary (`0`) periods.

In [ ]:
# A new block starts whenever direction changes within a recording.
data["direction_block"] = (
    data.groupby("recording")["direction"]
        .transform(lambda s: s.ne(s.shift()).cumsum())
)

# Show the sequence of blocks for one recording.
(data[data["recording"] == example_name]
 [["direction_block", "direction"]]
 .drop_duplicates()
 .reset_index(drop=True))

## 6. Accumulate ball rotation within each bout

To know how far the ball has rotated since the beginning of a bout, we add the successive `ball_yaw_deg` values.

We deliberately set the first sample of every bout to **0°**, so each bout has the same starting reference.

Mathematically, this is a cumulative sum within each direction block.

In [ ]:
# Set the first sample of each bout to zero, then cumulatively sum within each bout.
first_in_bout = (
    data["direction_block"].ne(data.groupby("recording")["direction_block"].shift())
)

increments = data["ball_yaw_deg"].copy()
increments.loc[first_in_bout] = 0.0

data["cumul_yaw"] = (
    increments.groupby([data["recording"], data["direction_block"]]).cumsum()
)

data[["time_s", "ball_yaw_deg", "direction", "direction_block", "cumul_yaw"]].head(12)

### Visualise one bout
The x-axis below is shifted so that the selected bout starts at `0 s`.

In [ ]:
example = data[data["recording"] == example_name].copy()
moving = example[example["direction"] != 0]
example_block = int(moving["direction_block"].iloc[0])

g = data[(data["recording"] == example_name) & (data["direction_block"] == example_block)].copy()
elapsed = g["time_s"] - g["time_s"].iloc[0]

plt.plot(elapsed, g["cumul_yaw"], linewidth=1.5)
plt.axhline(0, linewidth=0.8, linestyle="--")
plt.xlabel("Time since bout start (s)")
plt.ylabel("Cumulative ball yaw (deg)")
plt.title(f"Cumulative ball rotation — block {example_block}, direction {g['direction'].iloc[0]:g}")
plt.show()

## 7. Correct for stimulus direction

Raw ball rotation has opposite signs for the two stimulus directions. We want one variable where **positive always means that the fly performed the expected optomotor response**.

Because:

- `+1` = grating CW → correct fly turn CW → ball should rotate CCW (negative)
- `-1` = grating CCW → correct fly turn CCW → ball should rotate CW (positive)

we calculate:

\[
\mathrm{corrected\ cumulative\ yaw} = -\mathrm{cumulative\ yaw}\times\mathrm{direction}
\]

A positive corrected value therefore means net movement in the expected direction.

In [ ]:
data["correct_cumul_yaw"] = -data["cumul_yaw"] * data["direction"]

data[["direction", "cumul_yaw", "correct_cumul_yaw"]].head(12)

### Plot all direction bouts from one spatial frequency
Each line starts at zero. Positive values mean the net ball rotation is consistent with the expected optomotor response.

In [ ]:
sf_example = sorted(data["spatial_frequency"].unique())[0]
d = data[(data["spatial_frequency"] == sf_example) & (data["direction"] != 0)]

plt.figure(figsize=(10, 5))
for block, g in d.groupby("direction_block"):
    elapsed = g["time_s"] - g["time_s"].iloc[0]
    direction = int(g["direction"].iloc[0])
    plt.plot(elapsed, g["correct_cumul_yaw"], linewidth=1.0, alpha=0.8,
             label=f"Block {block}: {'CW' if direction == 1 else 'CCW'}")

plt.axhline(0, linewidth=0.8, linestyle="--")
plt.xlabel("Time since bout start (s)")
plt.ylabel("Direction-corrected cumulative yaw (deg)")
plt.title(f"Optomotor bouts at {sf_example:g} cycles/degree")
plt.legend(fontsize=8, ncol=2, frameon=False)
plt.show()

## 8. Reduce each bout to its total corrected angular displacement

For the final comparison, each moving bout becomes **one observation**: the final corrected cumulative yaw at the end of that bout.

In this experimental protocol, blocks **3–12** are the full-duration repeated moving bouts. Blocks 2 and 17 are shorter transition bouts, so we do not use them in the final comparison. Stationary (`0`) blocks are also excluded.

In [ ]:
summary_rows = []

for (recording, sf, block), g in data.groupby(["recording", "spatial_frequency", "direction_block"], sort=True):
    direction = float(g["direction"].iloc[0])
    summary_rows.append({
        "recording": recording,
        "spatial_frequency": float(sf),
        "direction_block": int(block),
        "direction": direction,
        "duration_s": float(g["time_s"].iloc[-1] - g["time_s"].iloc[0]),
        "total_corrected_displacement_deg": float(g["correct_cumul_yaw"].iloc[-1]),
        "included": (3 <= int(block) <= 12) and (direction != 0),
    })

bouts = pd.DataFrame(summary_rows)
analysis_bouts = bouts[bouts["included"]].copy()
analysis_bouts.head(10)

## 9. Final response magnitude across spatial frequencies

Each small point is one full stimulus bout. The larger point is the mean for that spatial frequency and the error bar is the **95% confidence interval of the mean** (using the Student *t* distribution).

The individual points are important: very large rotations can strongly influence the mean.

In [ ]:
from scipy.stats import t

fig, ax = plt.subplots(figsize=(9, 5.5))

# Individual bouts
for sf, g in analysis_bouts.groupby("spatial_frequency", sort=True):
    x = np.full(len(g), sf)
    ax.scatter(x, g["total_corrected_displacement_deg"], alpha=0.55, s=28)

# Mean and 95% CI
means = []
for sf, g in analysis_bouts.groupby("spatial_frequency", sort=True):
    y = g["total_corrected_displacement_deg"].to_numpy()
    mean = y.mean()
    if len(y) > 1:
        sem = y.std(ddof=1) / np.sqrt(len(y))
        ci = t.ppf(0.975, df=len(y)-1) * sem
    else:
        ci = np.nan
    means.append((sf, mean, ci))

means = pd.DataFrame(means, columns=["spatial_frequency", "mean", "ci95"])
ax.errorbar(means["spatial_frequency"], means["mean"], yerr=means["ci95"],
            fmt="o-", linewidth=1.5, capsize=4, label="Mean ± 95% CI")
ax.axhline(0, linewidth=1.0, linestyle="--")
ax.set_xlabel("Spatial frequency (cycles/degree)")
ax.set_ylabel("Total corrected angular displacement (deg)")
ax.set_title("Magnitude of the optomotor response")
ax.legend(frameon=False)
plt.show()

means

## 10. Response consistency: proportion of bouts in the expected direction

The displacement plot keeps information about **how much** the fly turned, but a few very large rotations might dominate it (subject to your data).

We can therefore ask a second, simpler question:

> **In what proportion of bouts did the fly show a net response in the expected direction?**

For each full bout:

- final corrected displacement `> 0°` → expected direction (`correct = True`)
- final corrected displacement `< 0°` → opposite direction (`correct = False`)

This deliberately ignores response magnitude. A `+2°` bout and a `+500°` bout both count as one bout in the expected direction.

If direction were unrelated to the stimulus, a useful descriptive reference is **0.5**: equal numbers of bouts in the expected and opposite directions.

In [ ]:
analysis_bouts["correct"] = analysis_bouts["total_corrected_displacement_deg"] > 0

proportion = (analysis_bouts.groupby("spatial_frequency", as_index=False)
              .agg(n_correct=("correct", "sum"),
                   n_bouts=("correct", "size")))
proportion["proportion_correct"] = proportion["n_correct"] / proportion["n_bouts"]

plt.figure(figsize=(9, 5.5))
plt.plot(proportion["spatial_frequency"], proportion["proportion_correct"], "o-", linewidth=1.5)
plt.axhline(0.5, linewidth=1.0, linestyle="--", label="0.5 reference")
plt.ylim(-0.03, 1.03)
plt.xlabel("Spatial frequency (cycles/degree)")
plt.ylabel("Proportion of bouts in expected direction")
plt.title("Consistency of the optomotor response")
plt.legend(frameon=False)
plt.show()

proportion

In [ ]:
analysis_bouts.to_csv("optomotor_bout_results.csv", index=False)
print("Saved: optomotor_bout_results.csv")

try:
    from google.colab import files
    files.download("optomotor_bout_results.csv")
except ImportError:
    print("The file is in the same folder as this notebook.")